In [ ]:
from pathlib import Path
import urllib.request

JAR_URL = "https://repository.cloudera.com/repository/cloudera-repos/Impala/ImpalaJDBC42/2.6.26.1031/ImpalaJDBC42-2.6.26.1031.jar"
JAR_NAME = "ImpalaJDBC42-2.6.26.1031.jar"
jar_path = Path.cwd() / JAR_NAME

if jar_path.exists():
    print(f"Jar already exists, skipping download: {jar_path}")
else:
    print(f"Downloading {JAR_NAME}...")
    urllib.request.urlretrieve(JAR_URL, jar_path)
    print(f"Downloaded to: {jar_path}")

In [ ]:
%pip install -q jaydebeapi JPype1 pyathena boto3

In [ ]:
import logging
import os
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(name)s | %(levelname)s | %(message)s",
    force=True,
)

PROJECT_ROOT = Path.cwd()
jar_name = "ImpalaJDBC42-2.6.26.1031.jar"
candidate_paths = [
    PROJECT_ROOT / jar_name,
    Path("/usr/lib/spark/jars") / jar_name,
]
JAR_PATH = next((str(p) for p in candidate_paths if p.exists()), None)
if not JAR_PATH:
    raise FileNotFoundError(
        f"{jar_name} not found. Run the download cell first or place it in one of: {candidate_paths}"
    )

os.environ["CLASSPATH"] = JAR_PATH

print(f"Project root: {PROJECT_ROOT}")
print(f"JAR path: {JAR_PATH}")
print(f"Root logger level: {logging.getLogger().getEffectiveLevel()}")

In [ ]:
from connections import JDBCImpalaConnection, AthenaConnection, ConnectionManager
from runner import ValidationRunner

print("✓ Imports loaded")

In [ ]:
IMPALA_JDBC_URL = (
    "jdbc:impala://peanut-impala.cargill.com:21050;"
    "AuthMech=3;"
    "SSL=1;"
    "AllowSelfSignedCerts=1"
)

IMPALA_USERNAME = "ps696636@AP.CORP.CARGILL.COM"
IMPALA_PASSWORD = "WeRock@123"

ATHENA_REGION = "us-east-1"
ATHENA_S3_STAGING_DIR = "s3://amazon-sagemaker-438465132548-us-east-1-242038a46fe7/dzd_aow04z6eei0d47/6k5768rke0n2cn/dev/sys/athena/"
ATHENA_WORKGROUP = "workgroup-6k5768rke0n2cn-4csydjgqkcyhpj"

In [ ]:
conn_manager = ConnectionManager()

source_conn = JDBCImpalaConnection(
    jdbc_url=IMPALA_JDBC_URL,
    username=IMPALA_USERNAME,
    password=IMPALA_PASSWORD,
    jar_path=JAR_PATH,
)

target_conn = AthenaConnection(
    region_name=ATHENA_REGION,
    s3_staging_dir=ATHENA_S3_STAGING_DIR,
    workgroup=ATHENA_WORKGROUP,
)

conn_manager.set_source(source_conn)
conn_manager.set_target(target_conn)

print("Source and target connections initialized")

In [ ]:
source_result = conn_manager.source.execute("SELECT 1 AS ok")
print("Source smoke test result:", source_result)

In [ ]:
target_result = conn_manager.target.execute("SELECT 1 AS ok")
print("Target smoke test result:", target_result)

In [ ]:
table_configs = [
    {
        "source_database": "prd_internal_tc1",
        "source_table": "afko",
        "target_database": "minerva_dev_src_sap_cdp_tc1_prd_raw_db",
        "target_table": "afko",
        "count_check": {"is_enabled": True},
        "column_name_check": {"is_enabled": True, "skip_columns": ["updated_at"]},
        "not_null_check": {"is_enabled": True, "validation_list": ["aufnr", "xflag"]},
        "unique_check": {
            "is_enabled": True,
            "validation_list": ["aufnr", "xflag", ["aufnr", "xflag"]],
        },
    },
    {
        "source_database": "prd_internal_tc1",
        "source_table": "bseg",
        "target_database": "minerva_dev_src_sap_cdp_tc1_prd_raw_db",
        "target_table": "bseg",
        "count_check": {"is_enabled": True},
        "column_name_check": {"is_enabled": True, "skip_columns": ["updated_at"]},
        "column_datatype_check": {"is_enabled": True},
        "length_check": {"is_enabled": True},
        "not_null_check": {
            "is_enabled": True,
            "validation_list": ["mandt", "bukrs", "gjahr", "belnr", "buzei"],
        },
        "unique_check": {
            "is_enabled": True,
            "validation_list": ["gjahr", ["mandt", "bukrs", "gjahr", "belnr", "buzei"]],
        },
    },
]

print("✓ Table configs loaded")
print(f"Tables to validate: {len(table_configs)}")
for cfg in table_configs:
    print(f"  - {cfg['source_database']}.{cfg['source_table']}")

In [ ]:
# Run all validations
runner = ValidationRunner(
    source_connection=conn_manager.source,
    target_connection=conn_manager.target,
    executed_by="ps696636@AP.CORP.CARGILL.COM",
)

results = runner.run(table_configs)

# Save results to JSON
json_file = runner.save_to_json(results, output_dir="validation_results")

# Print summary
runner.print_summary(results)

In [ ]:
conn_manager.close_all()
print("All connections closed")